# 09 OCR scoring (M2-2): 20-plate timing sample, one engine per fresh runtime
Runs on Colab, **fresh runtime**, CPU. Plan: PLAN.md (M2, decisions of 2026-11-15). Set `ENGINE` in the parameter cell, run, then restart the runtime for the next engine.

**This notebook only measures cost.** No score is computed. It never prints plate text; labels are used only to pick which crops to time. First run with `INSTALL = False` to see the package plan and wheel sizes, then set `INSTALL = True`.

In [ ]:
# Parameters
ENGINE = 'paddle'   # 'paddle' | 'easyocr' | 'fastplate'  (one per fresh runtime)
INSTALL = False     # False: only print the install plan and wheel sizes
SEED = 0
N_SAMPLE, REPEATS = 20, 3
DRIVE = '/content/drive/MyDrive/alpr'
LOCAL = '/content/tars'
MD5 = {   # recorded in PLAN.md
    'us_crops.zip': 'c43e2910b0042d93f5de9f0d1fca1bf2',
    'us_text_labels.csv': 'c82118867fa7256d4baab25d54537255',
    'uc3m_yolo.tar': 'e3ef306731f33095bac983338c6adb04',
    'uc3m_labels.csv': 'deccc9443b71c73fe247666d8466527a',
}
PKGS = {'paddle': 'paddleocr==3.7.0 paddlepaddle==3.3.1', 'easyocr': 'easyocr', 'fastplate': 'fast-plate-ocr[onnx]'}
FASTPLATE_MODEL = 'cct-s-v2-global-model'
# oneDNN off on CPU: paddlepaddle 3.3.x fails in the oneDNN path (PaddlePaddle issue 77340); pins stay as in uv.lock
PADDLE_KWARGS = dict(lang='en', use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False, enable_mkldnn=False)

In [ ]:
import csv, hashlib, io, json, os, platform, random, re, shutil, statistics, subprocess, sys, tarfile, time, urllib.request, zipfile
from pathlib import Path
import numpy as np
from google.colab import drive
from PIL import Image

In [ ]:
# Install plan (nothing is installed here): packages uv would add and their wheel sizes from PyPI.
subprocess.run('pip install -q uv', shell=True, check=True)  # approved raw pip: only to bootstrap uv on Colab
r = subprocess.run(f'uv pip install --system --dry-run {PKGS[ENGINE]}', shell=True, capture_output=True, text=True)
pk = re.findall(r'^\s*\+ ([A-Za-z0-9_.\-]+)==(\S+)', r.stdout + r.stderr, re.M)
tag = f'cp{sys.version_info.major}{sys.version_info.minor}'
def wheel_mb(name, ver):
    try:
        files = json.load(urllib.request.urlopen(f'https://pypi.org/pypi/{name}/{ver}/json', timeout=30))['urls']
    except Exception:
        return None
    whl = [f for f in files if f['packagetype'] == 'bdist_wheel']
    pick = [f for f in whl if 'none-any' in f['filename']] or [f for f in whl if tag in f['filename'] and 'manylinux' in f['filename'] and 'x86_64' in f['filename']]
    return pick[0]['size'] / 1e6 if pick else None
sizes = {n: wheel_mb(n, v) for n, v in pk}
for n, v in pk:
    print(f'{n}=={v}: ' + (f'{sizes[n]:.1f} MB' if sizes[n] is not None else 'size unknown'))
print(f'{len(pk)} packages, known wheel sizes total {sum(s for s in sizes.values() if s):.0f} MB, unknown: {sum(s is None for s in sizes.values())}')
if not pk:
    print(r.stdout[-1500:], r.stderr[-1500:])
if ENGINE == 'paddle':
    print('PaddleOCR arguments:', PADDLE_KWARGS)

In [ ]:
assert INSTALL, 'read the plan above, then set INSTALL = True and run again'
subprocess.run('cd /content && (git -C vehicle-plate-recognition pull -q --ff-only || git clone -q https://github.com/medrees-1000/vehicle-plate-recognition.git)', shell=True, check=True)
subprocess.run(f'uv pip install --system -q --no-deps -e /content/vehicle-plate-recognition && uv pip install --system -q {PKGS[ENGINE]}', shell=True, check=True)

In [ ]:
from alpr import ccpd, uslabels
if ENGINE == 'paddle':
    from paddleocr import PaddleOCR
elif ENGINE == 'easyocr':
    import easyocr
else:
    from fast_plate_ocr import LicensePlateRecognizer

In [ ]:
drive.mount('/content/drive')
cpu = next((l.split(':', 1)[1].strip() for l in open('/proc/cpuinfo') if l.startswith('model name')), platform.processor())
print('CPU:', cpu, '| cores:', os.cpu_count(), '| python', platform.python_version(), '| engine', ENGINE)

In [ ]:
# Engine as declared in PLAN.md (decision log 2026-11-15). Model files downloaded during init are measured by mtime.
def new_mb(t0):
    skip = [Path.home() / '.cache' / d for d in ('uv', 'pip')]
    tot = 0
    for top in (Path.home() / '.paddlex', Path.home() / '.EasyOCR', Path.home() / '.cache'):
        for f in top.rglob('*') if top.exists() else []:
            if f.is_file() and not any(s in f.parents for s in skip) and f.stat().st_mtime >= t0:
                tot += f.stat().st_size
    return tot / 1e6

t0 = time.time(); t1 = time.perf_counter()
if ENGINE == 'paddle':
    eng = PaddleOCR(**PADDLE_KWARGS)
    run = lambda a: eng.predict(a[:, :, ::-1])
elif ENGINE == 'easyocr':
    eng = easyocr.Reader(['en'], gpu=False)
    run = lambda a: eng.readtext(a)
else:
    eng = LicensePlateRecognizer(FASTPLATE_MODEL)
    run = lambda a: eng.run(a)
print(f'init {time.perf_counter() - t1:.0f} s | new model files since start: {new_mb(t0):.1f} MB')

def spread(v):
    q = statistics.quantiles(v, n=4)
    return f'median {statistics.median(v) * 1000:.0f} ms | min {min(v) * 1000:.0f} | IQR {q[0] * 1000:.0f}-{q[2] * 1000:.0f} | max {max(v) * 1000:.0f}'

def time_ocr(arrs):
    run(arrs[0])   # warm-up, not counted
    ts = []
    for a in arrs:
        for _ in range(REPEATS):
            t = time.perf_counter(); run(a); ts.append(time.perf_counter() - t)
    return ts

In [ ]:
# US crops: labels MD5 is asserted on load; labels only pick which crops to time (no text is read or printed).
lab = Path(DRIVE, 'archives', 'us_text_labels.csv')
assert lab.exists(), 'us_text_labels.csv not on Drive'
assert ccpd.file_md5(lab) == MD5['us_text_labels.csv'], 'us_text_labels.csv MD5 differs from PLAN.md: stop and report'
Path(LOCAL).mkdir(exist_ok=True)
zsrc, zdst = Path(DRIVE, 'archives', 'us_crops.zip'), Path(LOCAL, 'us_crops.zip')
shutil.copyfile(zsrc, zdst)
assert ccpd.file_md5(zdst) == MD5['us_crops.zip'], 'us_crops.zip MD5 differs from PLAN.md'
with open(lab, newline='', encoding='utf-8') as f:
    readable = sorted(int(r['crop_no']) for r in csv.DictReader(f) if r['final_status'] == 'readable')
pick = random.Random(SEED).sample(readable, N_SAMPLE)
with zipfile.ZipFile(zdst) as z:
    idx = {int(r['crop_no']): r for r in csv.DictReader(io.StringIO(z.read('index.csv').decode()))}
    arrs = []
    for n in pick:
        png = z.read(f'crops/{n:04d}.png')
        assert hashlib.md5(png).hexdigest() == idx[n]['md5'], f'crop {n}: MD5 differs from index.csv'
        arrs.append(np.asarray(Image.open(io.BytesIO(png)).convert('RGB')))
print(f'{len(arrs)} readable US crops, widths px: median {statistics.median(a.shape[1] for a in arrs):.0f}')
us_t = time_ocr(arrs)
print(f'{ENGINE} on US crops: {spread(us_t)} | limit 500 ms')

In [ ]:
# UC3M crops (after the US sample): crop cost and OCR cost per plate. Only boxes are read; lp_id is never read or printed.
for fn in ('uc3m_labels.csv',):
    assert ccpd.file_md5(Path(DRIVE, 'archives', fn)) == MD5[fn], f'{fn} MD5 differs from PLAN.md'
tsrc, tdst = Path(DRIVE, 'archives', 'uc3m_yolo.tar'), Path(LOCAL, 'uc3m_yolo.tar')
t0 = time.perf_counter()
if not tdst.exists():
    shutil.copyfile(tsrc, tdst)
t1 = time.perf_counter()
assert ccpd.file_md5(tdst) == MD5['uc3m_yolo.tar'], 'uc3m_yolo.tar MD5 differs from PLAN.md'
print(f'uc3m tar: copy {t1 - t0:.0f} s, MD5 check {time.perf_counter() - t1:.0f} s')
with open(Path(DRIVE, 'archives', 'uc3m_labels.csv'), newline='', encoding='utf-8') as f:
    rows = [r for r in csv.DictReader(f) if r['split'] == 'test']
rows = random.Random(SEED).sample(rows, N_SAMPLE)
cropt, arrs2 = [], []
with tarfile.open(tdst) as t:
    by_stem = {Path(m.name).stem: m.name for m in t.getmembers() if m.isfile() and m.name.startswith('images/test/')}
    for r in rows:
        t0 = time.perf_counter()
        with Image.open(io.BytesIO(t.extractfile(by_stem[r['image_id']]).read())) as im:
            im = im.convert('RGB')
            assert im.size == (int(r['image_w']), int(r['image_h'])), 'image size differs from the labels file'
            box = uslabels.crop_box(tuple(float(r[k]) for k in ('cx', 'cy', 'w', 'h')), *im.size)
            arrs2.append(np.asarray(im.crop(box)))
        cropt.append(time.perf_counter() - t0)
print(f'UC3M crop (decode + crop) per plate: {spread(cropt)}')
print(f'crop widths px: median {statistics.median(a.shape[1] for a in arrs2):.0f}')
uc_t = time_ocr(arrs2)
print(f'{ENGINE} on UC3M crops: {spread(uc_t)} | limit 500 ms')